# CSV / Excel → Final Cut Pro XML

1~5번 셀을 위에서 아래로 실행합니다. 각 단계의 완료 표시를 확인한 뒤 다음 셀로 이동하세요. GPU는 필요하지 않습니다. 완성 영상이 아니라 편집 가능한 타임라인 초안을 만듭니다.

원본은 Google 실행 환경에 업로드됩니다. 민감한 원본은 Mac 로컬에서 처리하세요. 결과 ZIP에는 사용한 원본도 포함됩니다.

변환이 끝나면 Mac의 Final Cut Pro에서 XML을 가져와 장면·소리·자막을 확인하세요. 자동 검사만으로 최종 화면이나 가져오기 성공을 보장하지는 않습니다.

## 1. 실행 코드 준비

이 셀을 실행하면 필요한 코드와 패키지를 준비합니다. 완료 후 출력된 새 작업 폴더 경로를 확인하세요. 이 셀을 다시 실행하면 새로운 작업이 시작되므로 기획표와 미디어를 다시 업로드해야 합니다.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tempfile

CORE_REV = '4c46039009263ffc67fa606a61c0840daeb10054'
REPO_URL = 'https://github.com/Kongdataif/csv-to-fcpxml-starter.git'
repo = Path('/content') / ('fcpxml-code-' + CORE_REV)
if not repo.exists():
    repo.mkdir()
    subprocess.run(['git', 'init', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', REPO_URL], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', CORE_REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
if actual != CORE_REV:
    raise RuntimeError('코드 버전 불일치: 런타임을 삭제하고 다시 시작하세요.')
subprocess.run(['git', '-C', str(repo), 'diff', '--exit-code', 'HEAD', '--'], check=True)
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt')], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
for name in ('colab_support', 'run'):
    sys.modules.pop(name, None)
from colab_support import new_session, begin_upload, save_plan, save_media, require_plan, require_uploads, make_result_archive
from run import verified_result
project = new_session(Path('/content/fcpxml-sessions'))
print('코드 커밋:', CORE_REV)
print('새 작업 폴더:', project)


## 2. 기획표 한 개 업로드

CSV 또는 XLSX 한 개를 선택합니다. Numbers는 Excel로 내보내세요. CSV는 UTF-8로 저장합니다.

**기획표 저장 완료**와 장면 목록을 확인한 뒤 3번으로 이동합니다. 다시 선택하거나 취소하면 미디어 업로드와 변환을 다시 진행해야 합니다.

In [ ]:
from google.colab import files

begin_upload(project, 'plan')
with tempfile.TemporaryDirectory(prefix='fcpxml-upload-') as upload_dir:
    previous_cwd = Path.cwd()
    os.chdir(upload_dir)
    try:
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    rows = save_plan(project, uploaded)
print(f'기획표 저장 완료: {len(rows)}개 장면')
for i, row in enumerate(rows, 1):
    print(i, row['파일'], '|', row.get('화면 자막', ''))
print('내용이 맞으면 3번을 실행하세요. 수정 시 2번부터 다시 진행합니다.')


## 3. 사진·영상 업로드

기획표에 적은 원본을 모두 선택합니다. 파일명과 확장자를 유지하세요. 누락된 파일이 있으면 중단합니다. 미사용 추가 미디어는 제외합니다.

**미디어 저장 완료**가 출력된 다음 4번으로 이동합니다.

In [ ]:
require_plan(project)
begin_upload(project, 'media')
with tempfile.TemporaryDirectory(prefix='fcpxml-upload-') as upload_dir:
    previous_cwd = Path.cwd()
    os.chdir(upload_dir)
    try:
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    extras = save_media(project, uploaded)
require_uploads(project)
print('미디어 저장 완료:', project / 'Media')
if extras:
    print('기획표에 없어 제외한 파일:', ', '.join(extras))


## 4. 화면 설정 후 변환

LAYOUT: portrait=세로 1080×1920, landscape=가로 1920×1080, both=둘 다.
FIT: fit=전체 표시·여백 가능, fill=화면 채움·잘림 가능.

프로젝트는 30fps입니다. 화면 자막은 Basic Title입니다. 별도 SRT는 만들지 않습니다. 실패하면 원인을 수정하고 다시 실행하세요. 입력 변경 시 2번 또는 3번부터, 화면 설정만 변경 시 이 셀부터 진행합니다.

In [ ]:
LAYOUT = 'portrait'  # @param ['portrait', 'landscape', 'both']
FIT = 'fit'  # @param ['fit', 'fill']

require_uploads(project)
subprocess.run([sys.executable, str(repo / 'run.py'), str(project), '--layout', LAYOUT, '--fit', FIT], check=True)
report = verified_result(project)
print('변환·결과 검증 완료:', report['build_id'])
print('결과 폴더:', project / 'output')


## 5. 결과 ZIP 다운로드

이번 실행이 성공하고 입력·결과 파일이 그대로일 때만 다운로드합니다. ZIP에는 기획표, 사용한 원본 Media/, XML과 사진 MP4 캐시 output/이 들어갑니다.

Mac에서 ZIP을 새 폴더에 풀고 전체 구조를 유지합니다. Final Cut Pro 12.0 이상에서 파일 > 가져오기 > XML을 선택하고 output/portrait.fcpxml 또는 output/landscape.fcpxml을 가져옵니다. both이면 필요한 두 XML을 각각 가져옵니다.

XML만 옮기거나 output/media를 삭제하지 마세요. 사진 장면은 원본 사진이 아닌 해당 MP4 캐시를 참조합니다. 이전 Final Cut Pro 작업 폴더에 새 결과를 덮어쓰지 마세요.

In [ ]:
require_uploads(project)
archive = make_result_archive(project, project.parent / 'fcpxml-result.zip')
print('다운로드 준비 완료:', archive)
files.download(str(archive))
